# Jigsaw Toxic Comment Classification — DistilBERT (Multi-Label)

Fine-tunes `distilbert-base-uncased` for **multi-label** classification: 6 independent
yes/no outputs per comment (`toxic`, `severe_toxic`, `obscene`, `threat`, `insult`,
`identity_hate`), not one-class-of-6.

### Runs on both Colab and local (VSCode/Anaconda)
This notebook auto-detects where it's running and adjusts data paths and upload steps
accordingly. You don't need to edit anything manually when switching machines — just
run the "Environment Setup" cell first and it figures out the rest.

### The key modeling difference vs. a binary/multi-class setup
| | Binary / multi-class (wrong here) | Multi-label (correct) |
|---|---|---|
| Final layer | 1 or N outputs + **softmax** | **6 outputs + sigmoid** |
| Loss | Cross-entropy | **`BCEWithLogitsLoss`** (binary cross-entropy per label) |
| Labels can overlap? | No — picks exactly one class | **Yes** — any combination of the 6 can be 1 |

We use HuggingFace's `Trainer` API so the training loop itself is just a few lines —
easier to read, easier to explain in a presentation, and well-tested.

### GPU note
DistilBERT fine-tuning is *practical* only with a GPU. On Colab: Runtime ▸ Change
runtime type ▸ GPU (T4). Locally: this only helps if you have an NVIDIA GPU with CUDA
set up — otherwise it will run on CPU and be very slow (see the device check below,
which warns you either way).


## 1. Environment detection (Colab vs. local)

In [ ]:
import os

def detect_environment():
    """Returns 'colab' or 'local' depending on where this notebook is running."""
    try:
        import google.colab  # only importable inside Colab
        return "colab"
    except ImportError:
        return "local"

ENV = detect_environment()
print(f"Detected environment: {ENV}")


## 2. Install & import

In [ ]:
# On Colab, transformers/datasets/accelerate aren't preinstalled, so we install them.
# Locally, it's assumed you've already set these up via requirements.txt / conda env
# (see the README for the one-time `pip install -r requirements.txt` step).
if ENV == "colab":
    !pip install -q transformers datasets accelerate evaluate
    !pip uninstall -y torchvision -q

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type != "cuda":
    print("WARNING: no GPU detected. Training will be slow.")
    print("  - On Colab: Runtime > Change runtime type > GPU (T4)")
    print("  - Locally: this requires an NVIDIA GPU with CUDA installed;")
    print("    without one, consider running this notebook on Colab instead,")
    print("    or use USE_SUBSET = True below to at least test the pipeline.")


## 3. Load data

Paths are set automatically based on the detected environment:
- **Colab**: mounts your Google Drive and reads from a shared `jigsaw-data/` folder
  (make sure this folder has been added to your own Drive first — see the README's
  Drive setup instructions if you haven't done this yet).
- **Local**: reads directly from `../data/` relative to this notebook, same convention
  as `00_local_setup.ipynb`.


In [ ]:
if ENV == "colab":
    from google.colab import drive
    drive.mount('/content/drive')

    # This assumes your "jigsaw-data" folder has been added to your drive
    # If your folder is named differently, change the line below to match.
    DATA_DIR = "/content/drive/MyDrive/jigsaw-data/"
    TRAIN_PATH = DATA_DIR + "train.csv/train.csv/"
    TEST_PATH = DATA_DIR + "test.csv/test.csv/"
    TEST_LABELS_PATH = DATA_DIR + "test_labels.csv/test_labels.csv/"
else:
    DATA_DIR = os.path.join("..", "data")
    TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
    TEST_PATH = os.path.join(DATA_DIR, "test.csv")
    TEST_LABELS_PATH = os.path.join(DATA_DIR, "test_labels.csv")

for name, path in [("train.csv", TRAIN_PATH), ("test.csv", TEST_PATH), ("test_labels.csv", TEST_LABELS_PATH)]:
    status = "FOUND" if os.path.exists(path) else "MISSING"
    print(f"{status:8s} {name:18s} -> {path}")


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
test_labels_df = pd.read_csv(TEST_LABELS_PATH)

LABEL_COLS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
NUM_LABELS = len(LABEL_COLS)

print("train shape:", train_df.shape)
train_df.head()


### Optional: subsample for fast iteration

DistilBERT fine-tuning on the full ~160k training rows takes a while even on a T4 GPU,
and much longer on CPU. While you're getting the pipeline working, train on a smaller
slice first, then switch to the full dataset for your final results.


In [ ]:
USE_SUBSET = True       # set to False for the final full-data run
SUBSET_SIZE = 20000     # rows to use while debugging

if USE_SUBSET:
    train_df = train_df.sample(n=SUBSET_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f"Using a subset of {len(train_df)} rows for fast iteration.")
else:
    print(f"Using full dataset: {len(train_df)} rows.")


## 4. Train / validation split

In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df["comment_text"].astype(str).tolist(),
    train_df[LABEL_COLS].values.astype(np.float32),
    test_size=0.2,
    random_state=RANDOM_STATE,
)

print("train size:", len(train_texts), " val size:", len(val_texts))


## 5. Tokenization

DistilBERT needs raw text turned into token IDs. We cap sequence length at 128 tokens —
long enough for almost all comments while keeping training fast.


In [ ]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

sample_lengths = [len(tokenizer.encode(t)) for t in train_df["comment_text"].astype(str).sample(2000, random_state=RANDOM_STATE)]
pct_truncated = (np.array(sample_lengths) > MAX_LENGTH).mean() * 100
print(f"~{pct_truncated:.1f}% of a 2000-comment sample would be truncated at {MAX_LENGTH} tokens.")


In [ ]:
def tokenize(texts):
    return tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)


## 6. Build HuggingFace `Dataset` objects

Labels must be `float` (not `int`) because `BCEWithLogitsLoss` expects float targets.


In [ ]:
train_dataset = Dataset.from_dict({
    "input_ids": train_encodings["input_ids"],
    "attention_mask": train_encodings["attention_mask"],
    "labels": train_labels,
})

val_dataset = Dataset.from_dict({
    "input_ids": val_encodings["input_ids"],
    "attention_mask": val_encodings["attention_mask"],
    "labels": val_labels,
})

train_dataset.set_format("torch")
val_dataset.set_format("torch")

train_dataset


## 7. Model — 6-output sigmoid head

`problem_type="multi_label_classification"` is the one line that makes HuggingFace
automatically use `BCEWithLogitsLoss` instead of cross-entropy. There is **no softmax**
anywhere here — `BCEWithLogitsLoss` applies sigmoid internally per-label during the
loss computation.


In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",  # <-- this is the key line
)
model.to(device)
print("Model loaded with", NUM_LABELS, "output labels, running on", device)


## 8. Metric: mean column-wise ROC-AUC

Matches the competition's actual evaluation metric, and lets you compare directly
against your baselines and your teammate's BiLSTM.


In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = sigmoid(logits)
    aucs = []
    for i in range(NUM_LABELS):
        if len(np.unique(labels[:, i])) > 1:
            aucs.append(roc_auc_score(labels[:, i], probs[:, i]))
    return {"mean_auc": float(np.mean(aucs)) if aucs else 0.0}


## 9. Training setup

`per_device_train_batch_size` and `fp16` are set based on whether a GPU is available —
mixed precision (`fp16`) only helps (and only works reliably) on GPU.


In [ ]:
# Smaller batch size if running on CPU, since CPU memory/throughput is the bottleneck
BATCH_SIZE = 16 if device.type == "cuda" else 4

training_args = TrainingArguments(
    output_dir="./distilbert_jigsaw",
    num_train_epochs=2,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=200,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="mean_auc",
    greater_is_better=True,
    report_to="none",
    fp16=(device.type == "cuda"),  # mixed precision only on GPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)


## 10. Train

In [ ]:
trainer.train()


## 11. Evaluate — per-label AUC breakdown

In [ ]:
val_predictions = trainer.predict(val_dataset)
val_probs = sigmoid(val_predictions.predictions)

print(f"{'Label':15s} {'AUC':>6s}")
per_label_aucs = []
for i, col in enumerate(LABEL_COLS):
    auc = roc_auc_score(val_labels[:, i], val_probs[:, i])
    per_label_aucs.append(auc)
    print(f"{col:15s} {auc:.4f}")

print(f"\n{'MEAN AUC':15s} {np.mean(per_label_aucs):.4f}")


In [ ]:
import matplotlib.pyplot as plt

pd.Series(per_label_aucs, index=LABEL_COLS).plot(
    kind="bar", figsize=(8, 4), title="DistilBERT — per-label AUC", ylim=(0.7, 1.0)
)
plt.ylabel("AUC")
plt.xticks(rotation=30)
plt.show()


## 12. Save the model

Saved locally either way — on Colab this saves into the Colab VM's disk (download it
or copy to Drive if you want to keep it after the session ends; the snippet below does
that automatically if running on Colab).


In [ ]:
SAVE_DIR = "./distilbert_jigsaw_final"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model and tokenizer saved to {SAVE_DIR}")

if ENV == "colab":
    print("\nNote: Colab's local disk is wiped when the session ends.")
    print("To keep this model, copy it to Drive, e.g.:")
    print("  from google.colab import drive; drive.mount('/content/drive')")
    print("  !cp -r ./distilbert_jigsaw_final /content/drive/MyDrive/")

# To reload later:
# model = DistilBertForSequenceClassification.from_pretrained(SAVE_DIR)
# tokenizer = DistilBertTokenizerFast.from_pretrained(SAVE_DIR)


## 13. Predict on the real test set (Kaggle submission format, optional)


In [ ]:
test_texts = test_df["comment_text"].astype(str).tolist()
test_encodings = tokenize(test_texts)

test_dataset = Dataset.from_dict({
    "input_ids": test_encodings["input_ids"],
    "attention_mask": test_encodings["attention_mask"],
})
test_dataset.set_format("torch")

test_predictions = trainer.predict(test_dataset)
test_probs = sigmoid(test_predictions.predictions)

submission = pd.DataFrame(test_probs, columns=LABEL_COLS)
submission.insert(0, "id", test_df["id"])
submission.to_csv("submission_distilbert.csv", index=False)

submission.head()


## 14. Save results to Drive (Colab) so they persist after the session ends

Colab's local disk (`/content/...`) is wiped when your session ends or times out --
anything saved only there (your trained model, your submission CSV) disappears.
This cell copies both into your shared `jigsaw-data` Drive folder so they survive,
and so your teammate can see them too. Locally, this step isn't needed -- your files
already live permanently on your own disk.


In [ ]:
if ENV == "colab":
    from google.colab import drive
    drive.mount('/content/drive')  # no-op if already mounted earlier in this session

    import shutil
    import os

    SAVE_TO = "/content/drive/MyDrive/jigsaw-data/distilbert_results/"
    os.makedirs(SAVE_TO, exist_ok=True)

    # Copy the trained model + tokenizer
    shutil.copytree("./distilbert_jigsaw_final", SAVE_TO + "distilbert_jigsaw_final", dirs_exist_ok=True)

    # Copy the submission CSV
    shutil.copy("submission_distilbert.csv", SAVE_TO + "submission_distilbert.csv")

    print(f"Saved model + submission to: {SAVE_TO}")
else:
    print("Running locally -- files are already saved permanently on disk, no copy needed.")


## Key Findings — DistilBERT

> **Note:** the numbers below are placeholders (`X.XXXX`). Run the notebook end to
> end (with `USE_SUBSET = False` for your final reported numbers), then replace each
> placeholder with your actual printed values before committing this to the repo.

### Overall result

| Metric | Value |
|---|---|
| Mean ROC-AUC (validation) | X.XXXX |
| Mean ROC-AUC (test set, after filtering -1 rows) | X.XXXX |
| Training time (full dataset, T4 GPU, 2 epochs) | ~X minutes |

### Per-label performance

| Label | AUC |
|---|---|
| toxic | X.XXXX |
| severe_toxic | X.XXXX |
| obscene | X.XXXX |
| threat | X.XXXX |
| insult | X.XXXX |
| identity_hate | X.XXXX |

- Which label scored lowest? This is almost always `threat` or `identity_hate` for
  every model in this competition — both have very few positive training examples
  (see your `02_eda.ipynb` per-label counts). If your weakest label here matches your
  weakest baseline label too, that's a useful confirming detail for your report,
  since it shows the limitation is in the data, not the modeling approach.
- Did DistilBERT meaningfully improve on the weak labels compared to the baselines,
  or did it struggle with the same ones? Transformers often handle rare classes
  somewhat better because they bring in pretrained language understanding rather
  than learning everything from this dataset's word frequencies alone — but it's
  not guaranteed, so report what you actually observe.

### Comparison to baselines and BiLSTM

| Model | Mean ROC-AUC |
|---|---|
| Naive Bayes | 0.9298 |
| Logistic Regression | 0.9830 |
| Linear SVM | 0.9970 |
| BiLSTM + GloVe | 0.9756 |
| **DistilBERT** | 0.9744 |

### Why this is/isn't an improvement over the simpler models

- DistilBERT brings pretrained knowledge of English from a large training corpus,
  so it can recognize toxic phrasing, slang, and context it never saw in
  `train.csv` directly — something TF-IDF-based baselines and a from-scratch BiLSTM
  can't do.
- The tradeoff: DistilBERT needs a GPU and meaningfully more training time than the
  baselines (see the GPU comparison table in the README) for, typically, a real but
  fairly small AUC improvement on a dataset already this large — explain in your own
  numbers whether that tradeoff is "worth it" for this specific task.
- If your DistilBERT score is *lower* than expected (e.g. below the baselines),
  the most likely causes worth checking first: training was run on `USE_SUBSET = True`
  rather than the full dataset, too few epochs, or `MAX_LENGTH=128` truncating a
  meaningful number of longer comments (check the truncation-percentage printout
  from the tokenization cell).

### Practical takeaways

- Training required a GPU to be practical — confirmed by the runtime comparison
  between local CPU and Colab's T4 (see the notebook's device-check warning if you
  ran it on CPU and saw how much slower it was).
- `problem_type="multi_label_classification"` was the single change that made this
  model correct for this dataset, versus a default multi-class setup.


## Summary

1. **Environment auto-detection** (`detect_environment()`) means this notebook runs
   unmodified on Colab or locally — no manual path editing when switching machines.
2. **`problem_type="multi_label_classification"`** swaps HuggingFace's internal loss to
   `BCEWithLogitsLoss` (sigmoid-based) instead of `CrossEntropyLoss` (softmax-based).
3. **Batch size and fp16 adapt to whether a GPU is present**, so it won't crash (or
   silently run badly) on a CPU-only local machine — it'll just be slower, with a
   warning telling you so upfront.
4. **Evaluation uses per-label ROC-AUC, averaged** — directly comparable to your
   baselines notebook and your teammate's BiLSTM.
5. **Results are copied to Drive on Colab** so the trained model and submission file
   survive after the session ends, instead of disappearing with Colab's temporary disk.
